___
# <center>Associação entre duas variáveis categóricas</center>
___

## Aula 16

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * transformar documentos de processos em colunas de uma tabela, usando
   expressões regulares;
 * medir a cobertura de uma expressão regular e decidir se a variável serve;
 * montar uma tabela de dupla entrada e comparar a proporção dentro de cada linha;
 * aplicar o teste qui-quadrado de independência e o teste de duas proporções,
   e escrever o resultado em uma frase.

___
<div id="indice"></div>

## Índice

- [A base de processos do BTG](#base)
- [Expressões regulares](#regex)
- [Independência e contagens esperadas](#independencia)
- [Teste qui-quadrado na base](#qui)
- [Exercício, parte a](#exercicio)
- [Duas proporções](#proporcoes)
- [Exercício, parte b](#exercicio_b)

___
<div id="base"></div>

# A base de processos do BTG

O banco entregou duas coisas:

* uma **planilha** com 30 mil contratos de financiamento de veículo que viraram
  processo: o número CNJ, a situação do contrato, o valor financiado, a taxa e
  o valor que o banco pagou no processo, quando pagou;
* **47 mil PDFs**: em cada processo, a petição inicial do autor, a contestação
  do banco e, às vezes, um e-mail do escritório ou o acordo.

Para a aula sorteamos **800 processos** e transformamos os documentos em colunas.
Cada linha de `btg_aulas.csv` é um processo.

> **Sobre a base.** Os dados são do projeto da clínica com o BTG e estão cobertos
> por sigilo bancário e pelo termo de parceria. O arquivo `btg_aulas.csv` está no
> BlackBoard e **não pode ser compartilhado** fora da turma, nem subido para o
> GitHub, nem colado em ferramentas externas além das usadas em aula.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Baixe btg_aulas.csv do BlackBoard e deixe na mesma pasta deste notebook.
# No Colab: clique no ícone de pasta, à esquerda, e arraste o arquivo para lá.
processos = pd.read_csv("btg_aulas.csv")
processos.shape

In [ ]:
processos.columns.tolist()

As colunas vêm de três fontes:

| fonte | colunas | como foram construídas |
|---|---|---|
| planilha do banco | `status_contrato`, `modalidade`, `valor_financiado`, `taxa_contrato_pct`, `valor_pago`, `pagou` | já vinham prontas |
| número CNJ | `tribunal`, `ano_ajuizamento` | expressão regular |
| PDFs | `tem_inicial`, `tem_contestacao`, `juizado`, `valor_causa`, `taxa_citada_pct` | expressão regular |
| PDFs | `tese_principal`, `pede_dano_moral`, `qtd_pedidos` e outras | modelo de linguagem (aula 17) |

`texto_inicial` e `texto_pedidos` trazem o começo da petição inicial e a seção de
pedidos, para você testar os padrões.

In [ ]:
# o desfecho que vamos estudar: o banco pagou alguma coisa no processo?
processos["pagou"].value_counts()

In [ ]:
processos["pagou"].mean()

[Volta ao Índice](#indice)

___
<div id="regex"></div>

# Expressões regulares

Uma expressão regular serve quando o pedaço de texto que você procura **tem
sempre a mesma forma**. O melhor exemplo é o número CNJ: `NNNNNNN-DD.AAAA.J.TR.OOOO`.
O `J` igual a 8 indica a Justiça Estadual, e o `TR` diz qual tribunal: 26 é São
Paulo, 21 é Rio Grande do Sul, 13 é Minas Gerais.

In [ ]:
import re

CNJ = re.compile(r"\d{7}-\d{2}\.(\d{4})\.8\.(\d{2})")

exemplo = processos["cnj"].iloc[0]
m = CNJ.search(exemplo)
exemplo, m.group(1), m.group(2)

O grupo 1 é o ano do ajuizamento e o grupo 2 é o código do tribunal. A coluna
`tribunal` da base foi construída exatamente assim, com uma tabela que traduz o
código para a sigla.

In [ ]:
processos["codigo_tr"] = processos["cnj"].str.extract(r"\.8\.(\d{2})")[0]
pd.crosstab(processos["codigo_tr"], processos["tribunal"]).iloc[:6, :6]

Agora um campo mais difícil: o **valor da causa**, que a petição inicial costuma
declarar no fim ("dá-se à causa o valor de R$ 21.593,82").

In [ ]:
VALOR_CAUSA = re.compile(
    r"(?:valor\s+da\s+causa|d[áa]-se\s+[àa]\s+causa).{0,60}?R\$\s*([\d.]{1,12},\d{2})",
    re.I)

def valor_da_causa(texto):
    if not isinstance(texto, str):
        return None
    m = VALOR_CAUSA.search(texto)
    if not m:
        return None
    return float(m.group(1).replace(".", "").replace(",", "."))

iniciais = processos[processos["tem_inicial"]].copy()
iniciais["valor_causa_regex"] = (iniciais["texto_inicial"].fillna("") + " "
                                 + iniciais["texto_pedidos"].fillna("")).map(valor_da_causa)
iniciais["valor_causa_regex"].notna().mean()

A expressão encontrou o valor da causa em cerca de **metade** das iniciais. Isso
não é um defeito da expressão: é uma informação sobre a base. Muitas iniciais
escrevem o valor de outro jeito ("atribui-se à presente o valor de..."), e
algumas simplesmente não declaram.

**Para você:** abra três iniciais em que a expressão **não** encontrou o valor e
veja como ele está escrito. Dá para ampliar o padrão sem pegar valores errados?

In [ ]:
sem_valor = iniciais[iniciais["valor_causa_regex"].isna()]
for texto in sem_valor["texto_pedidos"].dropna().head(3):
    print(texto[-600:])
    print("=" * 80)

[Volta ao Índice](#indice)

___
<div id="independencia"></div>

# Independência e contagens esperadas

Duas variáveis categóricas são **independentes** quando a proporção de uma é a
mesma em todos os grupos da outra. Na tabela abaixo, inventada, 10% dos
processos têm pagamento nos dois tribunais.

In [ ]:
tabela_a = pd.DataFrame({"pagou": [30, 15], "não pagou": [270, 135]},
                        index=["TJSP", "TJRS"])
tabela_a

In [ ]:
# proporção dentro de cada linha
tabela_a.div(tabela_a.sum(axis=1), axis=0).round(3)

Agora mexemos em poucos processos, sem mudar os totais. As variáveis ainda são
independentes?

In [ ]:
tabela_b = pd.DataFrame({"pagou": [33, 12], "não pagou": [267, 138]},
                        index=["TJSP", "TJRS"])
tabela_c = pd.DataFrame({"pagou": [36, 9], "não pagou": [264, 141]},
                        index=["TJSP", "TJRS"])

for nome, t in [("B", tabela_b), ("C", tabela_c)]:
    print(nome, t.div(t.sum(axis=1), axis=0)["pagou"].round(3).to_dict())

A contagem **esperada** de cada célula, se as variáveis fossem independentes, é

$$E = \frac{\text{total da linha} \times \text{total da coluna}}{n}$$

Como as três tabelas têm os mesmos totais, elas têm as mesmas esperadas.

In [ ]:
from scipy.stats import chi2_contingency

for nome, t in [("A", tabela_a), ("B", tabela_b), ("C", tabela_c)]:
    qui, valor_p, gl, esperado = chi2_contingency(t, correction=False)
    print(f"tabela {nome}: qui-quadrado = {qui:.2f}   valor-p = {valor_p:.3f}")

esperado

A estatística qui-quadrado soma, célula a célula, $(O - E)^2 / E$. Na tabela A
ela é zero, porque a tabela observada é igual à esperada. O valor-p diz a
probabilidade de uma tabela tão afastada da esperada quanto a observada
aparecer **por acaso**, se as variáveis fossem independentes.

* tabela B: valor-p de 0,32. Uma diferença de 11% contra 8% é comum por acaso.
* tabela C: valor-p de 0,046. Uma diferença de 12% contra 6% é pouco provável
  por acaso, com o corte usual de 5%.

[Volta ao Índice](#indice)

___
<div id="qui"></div>

# Teste qui-quadrado na base

A pergunta: **o tribunal e a ocorrência de pagamento são independentes?**
Primeiro a tabela e a proporção dentro de cada linha.

In [ ]:
pd.crosstab(processos["tribunal"], processos["pagou"]).sort_values(True, ascending=False).head(10)

São 24 tribunais, vários com poucos processos. O qui-quadrado precisa de umas 5
contagens **esperadas** por célula, e com 11,6% de pagamento um tribunal com 10
processos tem pouco mais de 1 esperado na coluna "pagou". Antes de testar,
agrupamos os tribunais raros. Essa decisão é sua, e precisa ser relatada.

In [ ]:
frequentes = processos["tribunal"].value_counts()
processos["tribunal_grupo"] = processos["tribunal"].where(
    processos["tribunal"].isin(frequentes[frequentes >= 40].index), "outros")

tabela = pd.crosstab(processos["tribunal_grupo"], processos["pagou"])
tabela

In [ ]:
(pd.crosstab(processos["tribunal_grupo"], processos["pagou"], normalize="index") * 100).round(1)

In [ ]:
from plotnine import ggplot, aes, geom_bar, labs, theme_minimal, coord_flip

(ggplot(processos, aes(x="tribunal_grupo", fill="pagou"))
 + geom_bar(position="fill")
 + coord_flip()
 + labs(x="", y="proporção dentro do tribunal", fill="pagou")
 + theme_minimal())

In [ ]:
qui, valor_p, gl, esperado = chi2_contingency(tabela)
print(f"qui-quadrado = {qui:.2f}   gl = {gl}   valor-p = {valor_p:.3f}")
print(f"menor contagem esperada: {esperado.min():.1f}")

**Como relatar:** nesta amostra de 800 processos, o tribunal e a ocorrência de
pagamento não são independentes ($\chi^2 = 9{,}71$; gl = 4; valor-p = 0,046). A
proporção de pagamento vai de 6,8% no TJBA a 22,2% no TJMG.

O teste não diz que o tribunal **causa** o pagamento: os tribunais recebem casos
diferentes. E o resultado depende do agrupamento: com outro corte, outro valor-p.

[Volta ao Índice](#indice)

___
<div id="exercicio"></div>

# Exercício, parte a: teste qui-quadrado

Em duplas. Complete as lacunas `____` e rode cada célula. Ao fim, escreva o
resultado **em uma frase**.

**O status do contrato está associado ao pagamento?**

In [ ]:
# 1. a tabela de dupla entrada: status do contrato nas linhas, pagou nas colunas
tabela = pd.crosstab(processos["____"], processos["pagou"])
tabela

In [ ]:
# 2. a proporção de pagamento DENTRO de cada status (dica: dentro de cada linha)
pd.crosstab(processos["status_contrato"], processos["pagou"], normalize="____").round(3)

In [ ]:
# 3. as contagens esperadas: alguma célula fica abaixo de 5?
qui, valor_p, gl, esperado = chi2_contingency(____)
esperado.round(1)

In [ ]:
# 4. "EM DIA" tem poucos processos: junte com "LIQUIDADO" (os dois sem atraso)
processos["status_grupo"] = processos["status_contrato"].replace({"EM DIA": "____"})
tabela = pd.crosstab(processos["status_grupo"], processos["pagou"])
qui, valor_p, gl, esperado = chi2_contingency(tabela)
print(tabela)
print(f"qui-quadrado = {qui:.2f}   gl = {gl}   valor-p = {valor_p:.3f}")

**Sua frase:** ____

[Volta ao Índice](#indice)

___
<div id="proporcoes"></div>

# Duas proporções

Quando as duas variáveis são binárias, dá para comparar diretamente as duas
proporções e, principalmente, calcular o **intervalo de confiança da diferença**.
A pergunta: a proporção de pagamento é a mesma nos processos com e sem
contestação?

In [ ]:
pd.crosstab(processos["tem_contestacao"], processos["pagou"], margins=True)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

sem = processos[~processos["tem_contestacao"]]["pagou"]
com = processos[processos["tem_contestacao"]]["pagou"]

z, valor_p = proportions_ztest([sem.sum(), com.sum()], [len(sem), len(com)])
baixo, alto = confint_proportions_2indep(sem.sum(), len(sem), com.sum(), len(com), method="wald")

print(f"sem contestação: {sem.mean():.3f}   com contestação: {com.mean():.3f}")
print(f"z = {z:.2f}   valor-p = {valor_p:.3f}")
print(f"diferença: {(sem.mean() - com.mean())*100:.1f} pontos, "
      f"de {baixo*100:.1f} a {alto*100:.1f} (IC 95%)")

In [ ]:
# numa tabela 2x2, o qui-quadrado sem correção dá o mesmo valor-p
t = pd.crosstab(processos["tem_contestacao"], processos["pagou"])
chi2_contingency(t, correction=False)[1]

**Como relatar:** a diferença estimada é de 5 pontos percentuais, com intervalo
de 95% de -0,9 a 10,9 pontos (valor-p = 0,07). **Estes dados não mostram
diferença** ao nível de 5%, mas são compatíveis tanto com nenhuma diferença
quanto com uma diferença de 10 pontos. Escrever "não há diferença" seria errado.

[Volta ao Índice](#indice)

___
<div id="exercicio_b"></div>

# Exercício, parte b: teste de duas proporções

Em duplas, como na parte a.

**A proporção de pagamento é a mesma nos processos com e sem petição inicial
entre os documentos que o banco guardou?**

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

com = processos[processos["tem_inicial"]]["pagou"]
sem = processos[~processos["____"]]["pagou"]
print(f"com inicial: {com.mean():.3f} (n = {len(com)})   sem inicial: {sem.mean():.3f} (n = {len(sem)})")

In [ ]:
z, valor_p = proportions_ztest([sem.sum(), com.sum()], [len(sem), len(____)])
baixo, alto = confint_proportions_2indep(sem.sum(), len(sem), com.sum(), len(com), method="wald")
print(f"z = {z:.2f}   valor-p = {valor_p:.6f}")
print(f"diferença: de {baixo*100:.1f} a {alto*100:.1f} pontos percentuais (IC 95%)")

**Sua frase:** ____

Cuidado: o teste mostra associação. Não escreva que ter a inicial no dossiê
**causa** alguma coisa.

[Volta ao Índice](#indice)